In [1]:
print("all ok")

all ok


https://github.com/confident-ai/deepeval

https://deepeval.com/

https://www.confident-ai.com/

In [2]:
#RAG Evaluation
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

In [ ]:
# Agent Evaluation
from deepeval.metrics import (
    # Reasoning / Planning
    PlanQualityMetric,
    PlanAdherenceMetric,

    # Tool / Action
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,

    # Overall Execution
    TaskCompletionMetric,
    StepEfficiencyMetric,
)

from deepeval.metrics import (
    ToolUseMetric,
    GoalAccuracyMetric,
)

from deepeval.metrics import (
    ConversationCompletenessMetric,
    ConversationRelevancyMetric,
    RoleAdherenceMetric,
    KnowledgeRetentionMetric,
)

In [4]:
# Safety Evaluation
from deepeval.metrics import (
    BiasMetric,
    ToxicityMetric,
    NonAdviceMetric,
    MisuseMetric,
    PIILeakageMetric,
    RoleViolationMetric,
)

In [5]:
# General LLM / Output Evaluation
from deepeval.metrics import (
    HallucinationMetric,
    PromptAlignmentMetric,
    JsonCorrectnessMetric,
    SummarizationMetric,
    ExactMatchMetric,
)

In [6]:
# Custom LLM-as-a-Judge
from deepeval.metrics import (
    GEval,
    DAGMetric,
    ArenaGEval,
)

In [7]:
from deepeval.test_case import (
    LLMTestCase,
    ToolCall,
    SingleTurnParams,
)
from deepeval.test_case import (
    ConversationalTestCase,
    Turn,
)
from deepeval.test_case import (
    ArenaTestCase,
    Contestant,
)
from deepeval.dataset import (
    EvaluationDataset,
    Golden,
)
from deepeval import (
    evaluate,
    assert_test,
)
from deepeval.tracing import (
    observe,
    update_current_span,
)

In [ ]:
# from deepeval import evaluate, assert_test

# from deepeval.dataset import (
#     EvaluationDataset,
#     Golden,
# )

# from deepeval.test_case import (
#     LLMTestCase,
#     ConversationalTestCase,
#     Turn,
#     ToolCall,
#     SingleTurnParams,
#     ArenaTestCase,
#     Contestant,
# )

# from deepeval.metrics import (
#     # RAG
#     AnswerRelevancyMetric,
#     FaithfulnessMetric,
#     ContextualRelevancyMetric,
#     ContextualPrecisionMetric,
#     ContextualRecallMetric,

#     # Agent
#     PlanQualityMetric,
#     PlanAdherenceMetric,
#     ToolCorrectnessMetric,
#     ArgumentCorrectnessMetric,
#     TaskCompletionMetric,
#     StepEfficiencyMetric,
#     ToolUseMetric,
#     GoalAccuracyMetric,

#     # Chatbot
#     ConversationCompletenessMetric,
#     ConversationRelevancyMetric,
#     RoleAdherenceMetric,
#     KnowledgeRetentionMetric,

#     # Safety
#     BiasMetric,
#     ToxicityMetric,
#     NonAdviceMetric,
#     MisuseMetric,
#     PIILeakageMetric,
#     RoleViolationMetric,

#     # General
#     HallucinationMetric,
#     PromptAlignmentMetric,
#     JsonCorrectnessMetric,
#     SummarizationMetric,
#     ExactMatchMetric,

#     # Custom
#     GEval,
#     DAGMetric,
#     ArenaGEval,
# )

# from deepeval.tracing import (
#     observe,
#     update_current_span,
# )

In [9]:
import os
import asyncio
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is missing. Please add it to the .env file."
    )

In [11]:
documents = [
    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={"id": "ragas_intro"},
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications.

Experiments can track evaluation results,
compare different application versions,
and store experiment outputs.
""".strip(),
        metadata={"id": "ragas_experiments"},
    ),

    Document(
        page_content="""
Ragas metrics measure different properties
of AI systems.

Examples include factual correctness,
faithfulness, answer relevancy,
retrieval quality, and agent behavior.
""".strip(),
        metadata={"id": "ragas_metrics"},
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge source and provides that
information to an LLM before generating
the final answer.
""".strip(),
        metadata={"id": "rag_intro"},
    ),
]

In [12]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

In [13]:
vector_store = InMemoryVectorStore(
    embedding=embedding_model,
)

vector_store.add_documents(documents)

['a51c0dbb-dd53-4bc6-a1c4-7faaeed56864',
 'cec19349-f417-483b-85e7-bdcdd8b3587a',
 '898e20e6-f560-4d83-8851-2711d97c0bcd',
 '4c4292c3-f680-42ec-a310-1714ae0df993']

In [14]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2},
)

In [15]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [16]:
rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the user's question using the provided context.

If the context does not contain enough information,
say that you do not have enough information.

CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""
)

In [18]:
rag_chain = (
    rag_prompt
    | llm
    | StrOutputParser()
)

In [19]:
async def rag_query(question):

    retrieved_documents = await retriever.ainvoke(question)

    retrieved_contexts = [
        document.page_content
        for document in retrieved_documents
    ]

    retrieved_document_ids = [
        document.metadata["id"]
        for document in retrieved_documents
    ]

    context = "\n\n".join(retrieved_contexts)

    answer = await rag_chain.ainvoke(
        {
            "context": context,
            "question": question,
        }
    )

    return {
        "answer": answer,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,
    }

In [20]:
question = "What is RAG and how does it work?"

In [22]:
await rag_query(question)

{'answer': "RAG stands for Retrieval-Augmented Generation. It is a system that retrieves relevant information from a knowledge source and provides that information to a large language model (LLM) before generating the final answer. This process enhances the LLM's ability to produce accurate and contextually relevant responses by incorporating external information.",
 'retrieved_contexts': ['RAG stands for Retrieval-Augmented Generation.\n\nA RAG system retrieves relevant information\nfrom a knowledge source and provides that\ninformation to an LLM before generating\nthe final answer.',
  'Ragas is an open-source evaluation framework designed\nfor evaluating and improving LLM applications.\n\nIt supports RAG evaluation, agent evaluation,\ncustom metrics, datasets, experiments,\nand synthetic test data generation.'],
 'retrieved_document_ids': ['rag_intro', 'ragas_intro']}

## Evaluation of the above app